# Lab 2: The Plan-and-Execute Newsroom

Build a multi-agent system that uses a Planner to decompose tasks and specialist agents (researcher, analyst, writer) to execute them. Uses `@observe` from Lab 1 for end-to-end observability.

## Phase 0: Setup and Imports

In [ ]:
import requests
import json
import os
import asyncio
import logging
from typing import List
from bs4 import BeautifulSoup
from litellm import acompletion
from pydantic import BaseModel, Field
from deepeval.tracing import observe
import nest_asyncio

nest_asyncio.apply()
from dotenv import load_dotenv
load_dotenv()

MODEL_NAME = os.getenv('MODEL_NAME', 'gpt-4o-mini')
print(f'Environment loaded. Model: {MODEL_NAME}')

### Tool Belt
Pre-built tools for live web search.

In [ ]:
def search_web(query, max_results=3):
    url = 'https://html.duckduckgo.com/html/'
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        resp = requests.post(url, data={'q': query}, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        results = []
        for res in soup.find_all('div', class_='result', limit=max_results):
            title_tag = res.find('a', class_='result__a')
            snippet_tag = res.find('a', class_='result__snippet')
            if title_tag and snippet_tag:
                results.append({'title': title_tag.get_text(strip=True), 'link': title_tag['href'], 'snippet': snippet_tag.get_text(strip=True)})
        return json.dumps(results)
    except Exception as e:
        return f'Search error: {e}'

print('Tool Belt loaded.')

## Phase 1: Specialist Prompts

Define the system prompts for each specialist. The **Researcher** has access to `search_web`.

In [ ]:
SPECIALIST_PROMPTS = {
    'researcher': 'You are a Research Specialist. Use the search_web tool to find real-time information and cite your sources.',
    'analyst': 'You are an Analysis Specialist. Evaluate the research, flag contradictions, and rate your confidence.',
    'writer': 'You are a Writing Specialist. Synthesize the analysis into a polished report with clear headings.',
}

SEARCH_TOOL_SCHEMA = [{
    'type': 'function',
    'function': {
        'name': 'search_web',
        'description': 'Search the web for real-time information.',
        'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
    }
}]

@observe(type='tool')
async def call_specialist(specialist, task):
    prompt = SPECIALIST_PROMPTS.get(specialist, 'You are a helpful assistant.')
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': task}
    ]
    tools = SEARCH_TOOL_SCHEMA if specialist == 'researcher' else None
    resp = await acompletion(model=MODEL_NAME, messages=messages, tools=tools)
    msg = resp.choices[0].message
    if specialist == 'researcher' and hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            if tc.function.name == 'search_web':
                args = json.loads(tc.function.arguments)
                result = search_web(args['query'])
                messages.append(msg.model_dump())
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
                final = await acompletion(model=MODEL_NAME, messages=messages)
                return final.choices[0].message.content
    return msg.content

print('Specialist system ready.')

## Phase 2: The Planner

The Planner decomposes a query into ordered specialist steps using structured output.

In [ ]:
class PlanStep(BaseModel):
    step: int = Field(..., description='Step number')
    task: str = Field(..., description='Task to perform')
    specialist: str = Field(..., description='One of: researcher, analyst, writer')
    depends_on: List[int] = Field(default_factory=list)

class Plan(BaseModel):
    steps: List[PlanStep]

PLANNER_PROMPT = '''You are a task planner for a newsroom.
Specialists: researcher (finds facts), analyst (evaluates), writer (synthesizes report).
Decompose the query into minimal ordered steps. researcher steps can run independently.
Query: {query}'''

class TaskPlanner:
    @observe(type='agent')
    async def create_plan(self, query):
        resp = await acompletion(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': PLANNER_PROMPT.format(query=query)}],
            response_format=Plan
        )
        plan = Plan.model_validate_json(resp.choices[0].message.content)
        return [s.model_dump() for s in plan.steps]

print('TaskPlanner ready.')

## Phase 3: The Newsroom Agent

Orchestrates planning and execution. Results from one step feed into the next as context.

In [ ]:
class NewsroomAgent:
    def __init__(self):
        self.planner = TaskPlanner()
        self.logger = logging.getLogger('Newsroom')
        logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')

    def _get_context(self, step, results):
        parts = [f'[Step {d} result]:\n{results[d]}' for d in step.get('depends_on', []) if d in results]
        return '\n\n'.join(parts)

    @observe(type='agent')
    async def run(self, query):
        self.logger.info('Planning...')
        plan = await self.planner.create_plan(query)
        for s in plan:
            self.logger.info(f'  Step {s["step"]} [{s["specialist"]}]: {s["task"]}')

        self.logger.info('Executing...')
        results = {}
        for step in plan:
            context = self._get_context(step, results)
            full_task = step['task'] if not context else f'{step["task"]}\n\nContext:\n{context}'
            self.logger.info(f'Running Step {step["step"]} [{step["specialist"]}]...')
            results[step['step']] = await call_specialist(step['specialist'], full_task)

        self.logger.info('Synthesizing...')
        summary = '\n'.join([f'Step {k}: {v}' for k, v in sorted(results.items())])
        final_resp = await acompletion(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': 'Write a final cited report from these research findings.'},
                {'role': 'user', 'content': f'Query: {query}\n\nFindings:\n{summary}'}
            ]
        )
        return final_resp.choices[0].message.content

print('NewsroomAgent ready.')

## Phase 4: Run the Agent

Change `query` to any research topic and run!

In [ ]:
async def main():
    agent = NewsroomAgent()
    query = 'Compare AI regulations in the EU and Saudi Arabia'
    report = await agent.run(query)
    print(f'\n--- FINAL REPORT ---\n{report}')

await main()